## VeriCirq - Examples

*Dima Fedoriaka, May 2026*

This notebook shows how to use VeriCirq library for formal verification of quantum circuits.

For an autonomous implementation and verification workflow (including Q# porting), see [.github/skills/vericirq-circuit-implementation/SKILL.md](.github/skills/vericirq-circuit-implementation/SKILL.md).

### Motivation

Consider a quantum circuit that implements an arithmetic operation. If its input is a a basis state (as opposed to superposition of basis state), its output is also a basis state. Such gates are sometimes called "permutation gates" because their unitary is a permutation matrix. Here we will talk only about these gates.

Examples of arithmetic operations are: addition, multiplication, division, modular operations (multiplication, division, exponentiation). These circuits play important role in quantum cryptanalysis: most modern research in this field is about designing sophisticated circuit to implement complicated arithmetic operation (e.g. modular multiplication) and then plugging it (as an "oracle") into order finding circuit. If the oracle is implemented correctly, order finding, when ran on actual quantum computer, will reverse an operation that is exponentially hard on classical computer.

These oracles can be very sophisticated. How do we know that we implemented them correctly? Usually we write unit tests using (sparse) simulators: apply circuit on some fixed (or random) input state, read out output and check that result matches classically computed value (which is usually very easy to compute). However, this approach doesn't guarantee that the circuit produces correct output on ALL states.

This library uses magic of Z3 solver to check that circuit produces correct output on all outputs!


### PermutationGate

To verify a circuit with VeriCirq, it must be implemented as Cirq gate. More specifically, it must be a subclass of
`vericirq.PermGate` which itself is subclass of `cirq.Gate`.

This gate must satisfy the following conditions:
 * It must only consist of X, CNOT and CCNOT gates.
 * (Implied by above) It's a permutation gate - it maps basis states to basis states.
 * It acts on a fixed number of qubits. Some of the first qubits are split in 1 or more little-endian unsigned
    integer registers, and remaining qubits are the "ancilla" qubits.

Recall that "ancilla" qubits are temporary qubits that are guaranteed to be in 0 state in the beginning, and that must be returned to this state in the end.



To define a gate, you need 3 things:
 * `input_sizes` - sizes of input registers.
 * (Optional) `ancilla_size` - number of ancilla qubits. If not specified, defaults to 0 (no anicllas).
 * Implementation of the gates by deomposing it into X, CNOT and CCNOT gates. For this, implement `_decompose_`
     method that takes list of qubits and returns iterator over gates. This is Cirq's convention.

After you defined a gate, use `vericirq.GateVerifier` to verify it against your formal specification (expressed as z3 arithmetic expressions). See examples below to see how.

### Example 1: bitwise negation

Let's consider very simple circuit that flips each bit in a 16-bit number. 
Recall that bit flip is done by quantum gate `X`.

In [1]:
import vericirq
import cirq

class BitwiseNegation(vericirq.PermutationGate):
    input_sizes = [10]

    def _decompose_(self, qubits):
        for q in qubits:
            yield cirq.X(q)

gate = BitwiseNegation()
ver = vericirq.GateVerifier(gate)

After you created an instance of `GateVerifier`, you can access input and output variables - these are symbolic Z3 (BitVec) variables
that correspond to inputs and outputs of this gate.

In [2]:
print(ver.input_vars)
print(ver.output_vars)

[in_0]
[out_0]


In [3]:
inp, out = ver.input_vars[0], ver.output_vars[0]
type(inp), type(out)

(z3.z3.BitVecRef, z3.z3.BitVecRef)

The specification of the gate is: for any input it applies bitwise negation to it. Let's verify it:

In [4]:
ver.verify_spec(out == ~inp)

OK!

What happens if we accidentally use wrong spec?

In [5]:
ver.verify_spec(out == -inp)

FAIL! Counterexample: input [12], output [1011].

verifier reports that on input 0 your gate produces output 1023, which is not what your spec requires (the spec requires -0=0).

### Example 2.1: Cuccaro Adder

Now let's consider real arithmetic circuit: Cuccaro addder, introduced in [this paper](https://arxiv.org/pdf/quant-ph/0410184). It takes two n-bit unsigned integers `a` and `b`, and adds `a` to `b` in place, so b becomes `(a+b)%(2^n)`.

In [6]:
import cirq
from cirq import CNOT, CCNOT
from typing import Sequence, Iterator
import z3

def maj(a, b, c):
    yield CNOT(c, a)
    yield CNOT(c, b)
    yield CCNOT(a, b, c)

def uma(a, b, c):
    yield CCNOT(a, b, c),
    yield CNOT(c, a)
    yield CNOT(a, b)

# https://arxiv.org/pdf/quant-ph/0410184
class CuccaroAdder(vericirq.PermutationGate):
    """Computes: (a, b, z) := (a, (a+b)%(2^n), z⊕(a+b)/(2^n))."""
    def __init__(self, n: int):
        self.n = n

    @property
    def input_sizes(self):
        # First addend, second addend, carry.
        return [self.n, self.n, 1]

    @property
    def ancilla_size(self):
        return 1
    
    def _decompose_(self, qubits: Sequence[cirq.Qid]) -> Iterator[cirq.OP_TREE]:
        n = self.n
        assert len(qubits) == 2*n+2
        a, b, z, c = qubits[0:n], qubits[n:2*n], qubits[2*n], qubits[2*n+1]
    
        yield from maj(c, b[0], a[0])
        for i in range(1, n):
            yield from maj(a[i-1], b[i], a[i])
        yield CNOT(a[n-1], z)
        for i in range(n-1, 0, -1):
            yield from uma(a[i-1], b[i], a[i])
        yield uma(c, b[0], a[0])


Before verifying circuit, we can run simulation on one input. The library includes a helper for this.

In [7]:
from vericirq.simulation import simulate_gate_on_inputs

gate = CuccaroAdder(3)
simulate_gate_on_inputs(gate, [2, 3, 0])

[2, 5, 0]

Now, let's do the verification! First, verify that second resgister contains the sum. We don't need to add `%2^n`, because our `z3.BitVec` variables are already modulo `2^n`.

In [8]:
gate = CuccaroAdder(16)
ver = vericirq.GateVerifier(gate)
a_in, b_in, z_in = ver.input_vars
a_out, b_out, z_out = ver.output_vars

ver.verify_spec(b_out == (a_in + b_in))

OK!

While this is most improtant property of the adder, it's not sufficient to ensure there are no bugs. We need to check 3 more properties.

Check that first register remains unchanged:

In [9]:
ver.verify_spec(a_out == a_in)

OK!

Check that `z` is flipped if and only if overflow happened:

In [10]:
overflow = z3.Not(z3.BVAddNoOverflow(a_in, b_in, False))
ver.verify_spec((z_in != z_out) == overflow)

OK!

Finally, check that all ancillas are returned in 0 state. For this, there is a special method:

In [11]:
ver.verify_ancillas()

OK!

If you write a unit test, you can collect all conditions together like this:

In [12]:
def verify_cuccaro_adder(adder):
    ver = vericirq.GateVerifier(adder)
    a_in, b_in, z_in = ver.input_vars
    a_out, b_out, z_out = ver.output_vars
    ver.verify_spec(a_out == a_in).assert_ok()
    ver.verify_spec(b_out == (a_in + b_in)).assert_ok()
    overflow = z3.Not(z3.BVAddNoOverflow(a_in, b_in, False))
    ver.verify_spec((z_in != z_out) == overflow).assert_ok()
    ver.verify_ancillas().assert_ok()
    ver.solver.reset()

verify_cuccaro_adder(CuccaroAdder(4))
verify_cuccaro_adder(CuccaroAdder(10))
print("OK")

OK


Note that `verify_cuccaro_adder` is a full formal spec for Cuccarro Adder - it specifies for every possible input what output in each register (including ancillas) must be.

In case you want to look exactly at circuit being verified, you can do that:

In [13]:
gate = CuccaroAdder(3)
ver = vericirq.GateVerifier(CuccaroAdder(3))
ver.circuit

0: ───@───@───X───X───────@───────────────────────────────@───X───@───X───@───────
      │   │   │   │       │                               │   │   │   │   │
1: ───┼───┼───┼───@───@───X───X───────@───────@───X───@───X───@───┼───┼───┼───────
      │   │   │       │   │   │       │       │   │   │   │       │   │   │
2: ───┼───┼───┼───────┼───┼───@───@───X───@───X───@───┼───┼───────┼───┼───┼───────
      │   │   │       │   │       │   │   │   │       │   │       │   │   │
3: ───┼───X───@───────┼───┼───────┼───┼───┼───┼───────┼───┼───────┼───@───┼───X───
      │       │       │   │       │   │   │   │       │   │       │   │   │   │
4: ───┼───────┼───────X───@───────┼───┼───┼───┼───────┼───@───────X───┼───┼───┼───
      │       │                   │   │   │   │       │               │   │   │
5: ───┼───────┼───────────────────X───@───┼───@───────X───────────────┼───┼───┼───
      │       │                           │                           │   │   │
6: ───┼───────┼───────────────────────────X───────────────────────────┼───┼───┼───
      │       │                                                       │   │   │
7: ───X───────@───────────────────────────────────────────────────────@───X───@───

### Example 2.2: Buggy Cuccaro Adder

Let's add a bug that affects carry bit computation.

In [14]:
class BuggyCuccaroAdder(vericirq.PermutationGate):
    def __init__(self, n: int):
        self.n = n

    @property
    def input_sizes(self):
        # First addend, second addend, carry.
        return [self.n, self.n, 1]

    @property
    def ancilla_size(self):
        return 1
    
    def _decompose_(self, qubits: Sequence[cirq.Qid]) -> Iterator[cirq.OP_TREE]:
        n = self.n
        assert len(qubits) == 2*n+2
        a, b, z, c = qubits[0:n], qubits[n:2*n], qubits[2*n], qubits[2*n+1]
    
        yield from maj(c, b[0], a[0])
        for i in range(1, n):
            yield from maj(a[i-1], b[i], a[i])
        yield CCNOT(a[n-1], b[n-1], z)  # Bug: used CCNOT instead of CNOT.
        for i in range(n-1, 0, -1):
            yield from uma(a[i-1], b[i], a[i])
        yield uma(c, b[0], a[0])


verify_cuccaro_adder(BuggyCuccaroAdder(16))

AssertionError: FAIL! Counterexample: input [36864, 36866, 0], output [36864, 8194, 0].

Verification failed. In the error message we see what condition failed, and a concrete counterexample we can use for debugging.

### Example 2.3: Comparison with exhaustive simulation

To achieve the same level of certainty (i.e. correct for all inputs) using "naive" approach (unit tests with simulation), we need to run tests on all possible inputs (a, b and z). This is going ot be very slow. Let's see how slow.


In [20]:
import random
import time

def verify_single_input(adder, a_in, b_in, z_in):
    a_out, b_out, z_out = simulate_gate_on_inputs(adder, [a_in, b_in, z_in])
    assert a_out == a_in
    assert b_out == (a_in+b_in)%(2**n)
    assert z_out == z_in ^ (a_in+b_in)//(2**n)


def verify_exhaustive(adder):
    n = adder.n
    for a_in in range(0, 2**n):
        for b_in in range(2**n):
            for z_in in [0,1]:
                verify_single_input(adder, a_in, b_in, z_in)
    
for n in list(range(1,11)) + [20, 30, 50, 100]:
    adder = CuccaroAdder(n)
    
    if n <= 4:
        t0 = time.time() 
        verify_exhaustive(adder)
        print(f"{n=}. Exhaustive simulation time: {time.time()-t0:.3f}s.")

    if n <= 5:
        t0 = time.time()
        a, b, z = random.randint(0, 2**n-1), random.randint(0, 2**n-1), random.randint(0, 1)
        verify_single_input(adder, a, b, z)
        print(f"{n=}. Single input simulation time: {time.time()-t0:.3f}s.")
    
    t0 = time.time() 
    verify_cuccaro_adder(adder)
    print(f"{n=}. VeriCirq time: {time.time()-t0:.3f}s.")

n=1. Exhaustive simulation time: 0.022s.
n=1. Single input simulation time: 0.001s.
n=1. VeriCirq time: 0.012s.
n=2. Exhaustive simulation time: 0.066s.
n=2. Single input simulation time: 0.002s.
n=2. VeriCirq time: 0.016s.
n=3. Exhaustive simulation time: 0.586s.
n=3. Single input simulation time: 0.005s.
n=3. VeriCirq time: 0.019s.
n=4. Exhaustive simulation time: 31.726s.
n=4. Single input simulation time: 0.062s.
n=4. VeriCirq time: 0.024s.
n=5. Single input simulation time: 1.134s.
n=5. VeriCirq time: 0.028s.
n=6. VeriCirq time: 0.020s.
n=7. VeriCirq time: 0.023s.
n=8. VeriCirq time: 0.024s.
n=9. VeriCirq time: 0.025s.
n=10. VeriCirq time: 0.025s.
n=20. VeriCirq time: 0.079s.
n=30. VeriCirq time: 0.152s.
n=50. VeriCirq time: 0.307s.
n=100. VeriCirq time: 1.055s.


Exhaustive simulation took 0.5s for $n=3$ and 31s for $n=4$.

VeriCirq took ~20ms for $n=10$ and 1s for $n=100$.

But what's even more impressive, for $n>=4$ verifying circuit on **single input** with simulator takes longer than verifying it on **all inputs** with VeriCirq.

### More examples

For more examples see [vericirq/examples](vericirq/examples). This folder contains some real arithmetic circuits and formal specifications for them. Those specs are checked in [vericirq/test_vericirq.py](vericirq/test_vericirq.py).